<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week3_cleaning_STARTER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> ⏸️ **Pause and Predict**
> Before writing any cleaning code: open your cleaning_log.md and write the decision for each missing value column FIRST. Impute or drop? Why? Then write the code to implement your decision.

In [ ]:
# ── Colab setup (skip if running locally) ────────────────────────────────────
import os
if not os.path.exists('requirements.txt'):
    !git clone https://github.com/internshipinabook/data-science-internshipinabook.git book
    %cd book
    !pip install -r requirements.txt -q
    print('Colab setup complete ✅')
else:
    print('Running locally ✅')


# Week 3 — Data Cleaning
### Week 3 — The Data Science Internship | Kova Analytics

> **STARTER notebook.**

The Data Science Internship Book 0 of 9 in the InternshipInABook™ Series

This repository contains the practical exercises from the book.

📖 Complete internship experience:

Workplace scenarios
Mentor guidance
Reflection exercises
Portfolio
checkpoints
Capstone projects
👉 Get the complete book: https://selar.com/al990ay7ux

🚀 Start Here
Welcome to The Data Science Internship.

If you are new to the InternshipInABook™ Series:

Run this setup notebook.
Complete Week 1.
Follow the weekly internship schedule.
Build your portfolio.
Complete the capstone project.
🔗 Repository: https://github.com/internshipinabook/data-science-internshipinabook

Run every cell top to bottom. Each cell prints ✅ if everything is working or ❌ with a fix instruction. Complete every fix before moving to Week 1.

---

## ⚠️ Common Mistakes

| Mistake | What goes wrong | Fix |
|---------|-----------------|-----|
| Cleaning without logging the decision | Three weeks later nobody — including you — knows what changed or why. Every change goes in the log. | cleaning_log.md: column name, issue, decision, justification — one row per change |
| Using the same imputation strategy for every column | Mean works for symmetric distributions. Median works for skewed. Mode works for categoricals. The method must match the data. | Check the distribution shape before choosing the imputation method |

## 🏆 Challenge Task

> 🎯 **Core Path**
> Run `df.isnull().sum()` and `df.dtypes` on your cleaned dataset. Does everything look as expected? Fix anything that does not.

> 🔬 **Advanced Path**
> Write a cleaning function that accepts a raw DataFrame and returns a cleaned one, applying all your Week 3 decisions. It should be callable from any future notebook.

## ✅ What You Can Do After This Week

- Classify missing values as MCAR, MAR, or MNAR and apply the appropriate strategy
- Apply the full cleaning pipeline: type conversion, deduplication, outlier detection, column renaming
- Maintain a cleaning log documenting every decision with its justification
- Preserve an unmodified backup (`df_original`) throughout all cleaning operations
- Save a cleaned dataset and verify it with a before-and-after summary

*Add `week3_cleaning.ipynb` to your internship portfolio.*

<a href="https://colab.research.google.com/github/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week3_cleaning_STARTER.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>&nbsp;&nbsp;
<a href="https://github.com/internshipinabook/data-science-internshipinabook/blob/main/notebooks/week3/week3_cleaning_STARTER.ipynb">
  <img src="https://img.shields.io/badge/View%20on-GitHub-181717?logo=github" alt="View on GitHub"/>
</a>

---

## 🖥️ How to Run This Notebook

| Platform | Instructions |
|----------|-------------|
| **Google Colab** | Click the badge above — free, no setup needed |
| **Local Jupyter** | `pip install -r requirements.txt` then `jupyter lab` |
| **VS Code** | Open `.ipynb` with the Jupyter extension installed |
| **GitHub** | Click "View on GitHub" above for a read-only preview |

> **STARTER notebook** — Complete the `# YOUR CODE HERE` cells. Check the SOLUTION notebook if stuck.

In [ ]:
# ── PLATFORM SETUP ───────────────────────────────────────────────────────────
# Run this cell first. It installs missing libraries in Google Colab.
# In a local environment, skip it if requirements.txt is already installed.

import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📦 Google Colab detected — installing libraries...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'pandas>=1.5', 'numpy>=1.23', 'matplotlib>=3.6',
                    'seaborn>=0.12', 'scikit-learn>=1.2'], check=True)
    print("✅ Libraries installed.")
else:
    print("💻 Running locally.")
    print("   If you see ImportError below, run: pip install -r requirements.txt")

## 📂 Dataset

The dataset is included in the course repository. No Kaggle account required.

**File:** `supermarket_sales.csv`  
**Source:** `https://raw.githubusercontent.com/internshipinabook/data-science-internshipinabook/main/data/supermarket_sales.csv`

Run the cell below to load it directly.

## 🔄 Adaptability — Use Your Own Dataset

This notebook uses `supermarket_sales.csv` but **every technique works on any tabular dataset**.

**To swap in your own data, change only these four lines:**
```python
DATA_FILE   = 'supermarket_sales.csv'  # ← your CSV filename
DATE_COLS   = ['Date', 'Time']          # ← columns to parse as datetime (or [])
TARGET_COL  = 'Total'                   # ← the column you want to analyse
GROUP_COLS  = ['Branch', 'Product line']# ← categorical columns for grouped analysis
```

**Works well with:** retail sales data, survey responses, HR records, student grades,
weather data, sports statistics — any dataset with rows of observations and mixed column types.

> 💡 **Tip:** If your dataset has no datetime columns, set `DATE_COLS = []` and skip Step 6.

---

## 🔄 Using a Different Dataset?

This notebook is written for `supermarket_sales.csv` but the techniques apply to any tabular dataset.

**To adapt:**
1. Change `DATA_FILE` to your filename
2. Update `TARGET_COL` to your column of interest (or remove if doing EDA only)
3. Update column name references in charts (e.g. `'Product line'` → your equivalent)
4. The cleaning and EDA patterns work on any pandas DataFrame

```python
# ── Adapt these for your dataset ─────────────────────────────────────────────
DATA_FILE  = 'supermarket_sales.csv'   # ← change to your file
DATE_COLS  = ['Date', 'Time']          # ← columns to parse as datetime
CAT_COLS   = ['Branch', 'Product line', 'Customer type', 'Payment']  # ← your categoricals
NUM_COLS   = ['Unit price', 'Quantity', 'Total', 'gross income']      # ← your numericals
```

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Libraries loaded')

## 1. Load raw. Create `df = df_raw.copy()`. Never modify `df_raw`.

In [ ]:
# YOUR CODE HERE


## 2. Fix Data Types (Date→datetime, Time→datetime, Branch→category)

In [ ]:
# YOUR CODE HERE


## 3. Missing Values — count, classify (MCAR/MAR/MNAR), handle

In [ ]:
# YOUR CODE HERE


## 4. Duplicates — check and decide

In [ ]:
# YOUR CODE HERE


## 5. Outlier Investigation — IQR, full row context, decision

In [ ]:
# YOUR CODE HERE


## 6. Standardise Column Names — lowercase, underscores

In [ ]:
# YOUR CODE HERE


## 7. Cleaning Log

In [ ]:
cleaning_log = []
# cleaning_log.append({'column':'...','issue':'...','action':'...','rows_affected':0,'risk':'Low'})
print(pd.DataFrame(cleaning_log).to_string(index=False))

## 8. Save

In [ ]:
# df_clean.to_csv('supermarket_sales_clean.csv', index=False)
# print(f'✅ {df_clean.shape}')

---
## ✅ Week 3 Complete
**Next:** `week4/*_STARTER.ipynb`

---
*github.com/InternshipInABook*